In [ ]:
from enum import StrEnum


class Player(StrEnum):
    JULIET = "Juliet"
    ROMEO = "Romeo"


PLAYER_NAME = Player.ROMEO
FORMATTED_DATE_OVERRIDE = None


from google.colab import drive
from pathlib import Path
import shutil
from dataclasses import dataclass
import logging
from datetime import datetime, timezone
import os


@dataclass
class Config:
    impersonator: str
    impersonatee: str


logging.basicConfig(
    format="%(asctime)s %(levelname)-8s %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

logger = logging.getLogger(__name__)

logger.info("Player %s is playing...", PLAYER_NAME)

logger.info("Mounting drive for player %s...", PLAYER_NAME)
# drive.mount('/content/drive', force_remount=True)
logger.info("Mounted drive for player %s", PLAYER_NAME)

SRC_DIR = Path("/content/data")
BASE_DIR = Path("/content/drive/MyDrive/guess-it")
QUESTIONS_DIR = BASE_DIR / "questions"
QUESTIONS_FILE_NAME = "questions.yaml"
ANSWERS_DIR = BASE_DIR / "answers"

DISTRIBUTABLE_DIR = BASE_DIR / "dist"
DISTRIBUTABLE_PATH = DISTRIBUTABLE_DIR / "quizz-0.1.0-py3-none-any.whl"

logger.info("Installing distributable for player: %s...", PLAYER_NAME)
!pip install $DISTRIBUTABLE_PATH -q
logger.info("Installed distributable for player: %s", PLAYER_NAME)

In [ ]:
def get_player_config(player_name: Player) -> Config:
    logger.info("Getting player config for player: %s", player_name)
    PLAYER_CONFIG_MAPPING = {
        Player.JULIET: Config(impersonator="juliet", impersonatee="romeo"),
        Player.ROMEO: Config(impersonator="romeo", impersonatee="juliet"),
    }

    return PLAYER_CONFIG_MAPPING[player_name]


def get_questions_file_path(
    questions_dir: str | None = None, questions_file_name: str | None = None
) -> Path:
    questions_dir = questions_dir or QUESTIONS_DIR
    questions_file_name = questions_file_name or QUESTIONS_FILE_NAME
    return questions_dir / questions_file_name


def populate_env_vars(
    player_config: Config,
    questions_file_path: str | Path,
    distributable_path: str | Path,
) -> None:
    %env IMPERSONATOR={player_config.impersonator}
    %env IMPERSONATEE={player_config.impersonatee}
    %env QUESTIONS_FILE_PATH={questions_file_path}
    %env DISTRIBUTABLE_PATH={distributable_path}


def get_formatted_date(today_utc: datetime | None = None) -> str:
    if today_utc is None:
        today_utc = datetime.now(tz=timezone.utc)
    return today_utc.strftime("%Y%m%d")


def build_timestamp_answers_dir(formatted_date: str | None) -> Path:
    formatted_date = formatted_date or get_formatted_date()
    return ANSWERS_DIR / formatted_date


def populate_env_vars_judge(
    formatted_date: str, answers_dir: str | None = None
) -> None:
    answers_dir = answers_dir or ANSWERS_DIR
    base_path = f"{answers_dir}/{formatted_date}"

    env_vars = {
        "PLAYER_1_ANSWERS_FILE_PATH": f"{base_path}/juliet_answers.yaml",
        "PLAYER_2_ANSWERS_FILE_PATH": f"{base_path}/romeo_answers.yaml",
        "PLAYER_1_IMPERSONATEE_ANSWERS_FILE_PATH": f"{base_path}/juliet_as_romeo_answers.yaml",
        "PLAYER_2_IMPERSONATEE_ANSWERS_FILE_PATH": f"{base_path}/romeo_as_juliet_answers.yaml",
    }

    for key, value in env_vars.items():
        os.environ[key] = value


def copy_to_dir(src_dir: Path | str, dest_dir: str | Path) -> None:
    QUESTIONS_FILE_NAME = "questions.yaml"
    src_dir = Path(src_dir)
    dest_dir = Path(dest_dir)
    for file in src_dir.iterdir():
        logger.info("Copying %s to %s", file, dest_dir)
        shutil.copy(file, dest_dir)

    src_questions_file_path = QUESTIONS_DIR / QUESTIONS_FILE_NAME
    dest_questions_file_path = dest_dir / QUESTIONS_FILE_NAME
    shutil.copy(src_questions_file_path, dest_questions_file_path)


def main() -> None:
    formatted_date = get_formatted_date()
    logger.info("Formatted date: %s", formatted_date)
    formatted_date = FORMATTED_DATE_OVERRIDE or formatted_date
    timestamp_answers_dir = build_timestamp_answers_dir(formatted_date)
    logger.info("Timestamp answers dir: %s", timestamp_answers_dir)
    timestamp_answers_dir.mkdir(parents=True, exist_ok=True)
    questions_file_path = get_questions_file_path()
    logger.info("Questions file path: %s", questions_file_path)
    player_config = get_player_config(PLAYER_NAME)
    logger.info("Player config: %s", player_config)
    populate_env_vars(player_config, questions_file_path, DISTRIBUTABLE_PATH)
    logger.info("Populated env vars for player: %s", PLAYER_NAME)
    logger.info("Running quizz-player for player: %s", PLAYER_NAME)
    !quizz-player --impersonator $IMPERSONATOR --impersonatee $IMPERSONATEE --questions-file-path $QUESTIONS_FILE_PATH
    logger.info("Ran quizz-player for player: %s", PLAYER_NAME)
    logger.info(
        "Copying files to %s for player: %s", timestamp_answers_dir, PLAYER_NAME
    )
    copy_to_dir(SRC_DIR, timestamp_answers_dir)
    logger.info("Copied files to %s for player: %s", timestamp_answers_dir, PLAYER_NAME)


main()

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
drive.mount("/content/drive", force_remount=True)

In [ ]:
formatted_date = get_formatted_date()
formatted_date = FORMATTED_DATE_OVERRIDE or formatted_date
populate_env_vars_judge(formatted_date)

logger.info("Running quizz-judge for player: %s", PLAYER_NAME)
!quizz-judge judge --player-1-name julieto --player-2-name romeo --questions-file-path $QUESTIONS_FILE_PATH --player-1-answers-file-path=$PLAYER_1_ANSWERS_FILE_PATH --player-2-answers-file-path=$PLAYER_2_ANSWERS_FILE_PATH --player-1-impersonator-answers-file-path=$PLAYER_1_IMPERSONATEE_ANSWERS_FILE_PATH --player-2-impersonator-answers-file-path=$PLAYER_2_IMPERSONATEE_ANSWERS_FILE_PATH
logger.info("Ran quizz-judge for player: %s", PLAYER_NAME)